In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Raíz del proyecto (el notebook vive en /notebooks)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

# Spotify: desde el CSV original
spotify = pd.read_csv(ROOT / "data" / "raw" / "spotify_dataset.csv")

# Grammy: desde la base de datos fuente
url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DATA_DB_USER"),
    password=os.getenv("DATA_DB_PASSWORD"),
    host="localhost",
    port=5433,
    database=os.getenv("GRAMMY_DB_NAME"),
)
engine = create_engine(url)
grammy = pd.read_sql("SELECT * FROM grammy_awards", engine)

print("Spotify:", spotify.shape)
print("Grammy :", grammy.shape)

Spotify: (114000, 21)
Grammy : (4810, 10)


In [2]:
EVIDENCE = ROOT / "docs" / "evidence"
EVIDENCE.mkdir(parents=True, exist_ok=True)

def structure_summary(df, name):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "pct_nulos": (df.isna().mean() * 100).round(2),
        "valores_distintos": df.nunique(),
    })
    summary.to_csv(EVIDENCE / f"profiling_{name}_structure.csv")
    print(f"=== {name}: {df.shape[0]} filas x {df.shape[1]} columnas ===")
    return summary

display(structure_summary(spotify, "spotify"))
display(structure_summary(grammy, "grammy"))

=== spotify: 114000 filas x 21 columnas ===


,dtype,nulos,pct_nulos,valores_distintos
Unnamed: 0,int64,0,0.0,114000
track_id,str,0,0.0,89741
artists,str,1,0.0,31437
album_name,str,1,0.0,46589
track_name,str,1,0.0,73608
popularity,int64,0,0.0,101
duration_ms,int64,0,0.0,50697
explicit,bool,0,0.0,2
danceability,float64,0,0.0,1174
energy,float64,0,0.0,2083


=== grammy: 4810 filas x 10 columnas ===


,dtype,nulos,pct_nulos,valores_distintos
year,int64,0,0.00,62
title,str,0,0.00,62
published_at,"datetime64[us, UTC]",0,0.00,4
updated_at,"datetime64[us, UTC]",0,0.00,10
category,str,0,0.00,638
nominee,str,6,0.12,4131
artist,str,1840,38.25,1658
workers,str,2190,45.53,2366
img,str,1367,28.42,1463
winner,bool,0,0.00,1


In [3]:
def save_show(obj, name, title):
    df = obj.to_frame() if isinstance(obj, pd.Series) else obj
    df.to_csv(EVIDENCE / f"profiling_{name}.csv")
    print(f"=== {title} ===")
    display(df)

# --- Unicidad y duplicados: Spotify ---
sp_unique = pd.DataFrame({
    "metrica": [
        "filas",
        "track_id distintos",
        "filas con track_id repetido",
        "pares (track_id, track_genre) repetidos",
        "filas duplicadas (todas las columnas menos el indice)",
        "(artists, track_name) con mas de un track_id",
    ],
    "valor": [
        len(spotify),
        spotify["track_id"].nunique(),
        int(spotify["track_id"].duplicated(keep=False).sum()),
        int(spotify.duplicated(subset=["track_id", "track_genre"]).sum()),
        int(spotify.drop(columns=["Unnamed: 0"]).duplicated().sum()),
        int(spotify.groupby(["artists", "track_name"])["track_id"].nunique().gt(1).sum()),
    ],
})
save_show(sp_unique, "spotify_uniqueness", "Spotify: unicidad")

# --- Unicidad y recuperabilidad del artista: Grammy ---
gr_unique = pd.DataFrame({
    "metrica": [
        "filas",
        "duplicados (year, category, nominee, artist)",
        "artist nulo",
        "artist y workers nulos a la vez",
        "nominee nulo",
    ],
    "valor": [
        len(grammy),
        int(grammy.duplicated(subset=["year", "category", "nominee", "artist"]).sum()),
        int(grammy["artist"].isna().sum()),
        int((grammy["artist"].isna() & grammy["workers"].isna()).sum()),
        int(grammy["nominee"].isna().sum()),
    ],
})
save_show(gr_unique, "grammy_uniqueness", "Grammy: unicidad y artistas recuperables")

# --- Contenido categórico ---
save_show(spotify["track_genre"].value_counts(), "spotify_genre", "Spotify: filas por genero")
save_show(spotify["time_signature"].value_counts().sort_index(), "spotify_time_signature", "Spotify: time_signature")
save_show(spotify["explicit"].value_counts(), "spotify_explicit", "Spotify: explicit")
save_show(grammy["category"].value_counts().head(15), "grammy_category_top15", "Grammy: 15 categorias mas frecuentes")
save_show(grammy.loc[grammy["artist"].isna(), "category"].value_counts().head(15),
          "grammy_artist_null_by_category", "Grammy: artist nulo por categoria (top 15)")

# --- Separadores de artistas (riesgo de integracion) ---
art_s = spotify["artists"].dropna()
art_g = grammy["artist"].dropna()
sep = pd.DataFrame(
    [
        ("spotify", "; (varios artistas)", int(art_s.str.contains(";", regex=False).sum())),
        ("grammy", "&", int(art_g.str.contains("&", regex=False).sum())),
        ("grammy", "Featuring / feat", int(art_g.str.contains("feat", case=False, regex=False).sum())),
        ("grammy", ",", int(art_g.str.contains(",", regex=False).sum())),
        ("grammy", " and ", int(art_g.str.contains(" and ", case=False, regex=False).sum())),
    ],
    columns=["fuente", "patron", "filas"],
)
save_show(sep, "artist_separators", "Separadores de artistas por fuente")

=== Spotify: unicidad ===


,metrica,valor
0,filas,114000
1,track_id distintos,89741
2,filas con track_id repetido,40900
3,"pares (track_id, track_genre) repetidos",450
4,filas duplicadas (todas las columnas menos el ...,450
5,"(artists, track_name) con mas de un track_id",4657


=== Grammy: unicidad y artistas recuperables ===


,metrica,valor
0,filas,4810
1,"duplicados (year, category, nominee, artist)",0
2,artist nulo,1840
3,artist y workers nulos a la vez,186
4,nominee nulo,6


=== Spotify: filas por genero ===


...

,count
category,
Song Of The Year,70
Record Of The Year,69
Album Of The Year,66
Best Opera Recording,64
Best Album Notes,63
Best Country Song,55
Best Instrumental Composition,55
Best New Artist,51
Best Historical Album,44


=== Grammy: artist nulo por categoria (top 15) ===


,count
category,
Song Of The Year,70
Best Opera Recording,64
Best Album Notes,63
Best Country Song,55
Best Instrumental Composition,55
Best New Artist,50
Best Historical Album,44
Best Chamber Music Performance,43
Best Instrumental Arrangement,39


=== Separadores de artistas por fuente ===


,fuente,patron,filas
0,spotify,; (varios artistas),30075
1,grammy,&,415
2,grammy,Featuring / feat,87
3,grammy,",",124
4,grammy,and,60


In [4]:
for n in ["grammy_artist_null_by_category", "artist_separators", "grammy_category_top15"]:
    print("===", n)
    print(pd.read_csv(EVIDENCE / f"profiling_{n}.csv").to_string(index=False))
    print()

=== grammy_artist_null_by_category
                      category  count
              Song Of The Year     70
          Best Opera Recording     64
              Best Album Notes     63
             Best Country Song     55
 Best Instrumental Composition     55
               Best New Artist     50
         Best Historical Album     44
Best Chamber Music Performance     43
 Best Instrumental Arrangement     39
   Best Orchestral Performance     36
          Best Classical Album     35
                Best Rock Song     33
        Best Recording Package     32
      Best Rhythm & Blues Song     32
       Best Choral Performance     30

=== artist_separators
 Unnamed: 0  fuente              patron  filas
          0 spotify ; (varios artistas)  30075
          1  grammy                   &    415
          2  grammy    Featuring / feat     87
          3  grammy                   ,    124
          4  grammy                and      60

=== grammy_category_top15
                         

In [5]:
num_cols = ["popularity", "duration_ms", "danceability", "energy", "key", "loudness", "mode",
            "speechiness", "acousticness", "instrumentalness", "liveness", "valence",
            "tempo", "time_signature"]
desc = spotify[num_cols].describe().T[["min", "25%", "50%", "75%", "max"]]
desc.to_csv(EVIDENCE / "profiling_spotify_numeric.csv")
print(desc.round(3).to_string())

unit = ["danceability", "energy", "valence", "speechiness", "acousticness", "instrumentalness", "liveness"]
checks = pd.DataFrame({
    "chequeo": ["popularity == 0", "duration_ms < 30 s", "duration_ms > 20 min",
                "tempo == 0", "loudness > 0", "campo 0-1 fuera de rango"],
    "filas": [
        int((spotify["popularity"] == 0).sum()),
        int((spotify["duration_ms"] < 30000).sum()),
        int((spotify["duration_ms"] > 1_200_000).sum()),
        int((spotify["tempo"] == 0).sum()),
        int((spotify["loudness"] > 0).sum()),
        int(((spotify[unit] < 0) | (spotify[unit] > 1)).any(axis=1).sum()),
    ],
})
checks["pct"] = (checks["filas"] / len(spotify) * 100).round(2)
checks.to_csv(EVIDENCE / "profiling_spotify_range_checks.csv", index=False)
print()
print(checks.to_string(index=False))

# Temporal: Grammy
years = sorted(grammy["year"].unique())
missing = sorted(set(range(min(years), max(years) + 1)) - set(years))
title_year = pd.to_numeric(grammy["title"].str.extract(r"\((\d{4})\)")[0], errors="coerce")
print()
print("Grammy años:", min(years), "a", max(years), "| distintos:", len(years), "| faltantes:", missing)
print("title coincide con year:", round(float((title_year == grammy["year"]).mean()) * 100, 2), "%")
print(grammy["published_at"].dt.date.value_counts().sort_index().to_string())


                     min         25%         50%         75%          max
popularity         0.000      17.000      35.000      50.000      100.000
duration_ms        0.000  174066.000  212906.000  261506.000  5237295.000
danceability       0.000       0.456       0.580       0.695        0.985
energy             0.000       0.472       0.685       0.854        1.000
key                0.000       2.000       5.000       8.000       11.000
loudness         -49.531     -10.013      -7.004      -5.003        4.532
mode               0.000       0.000       1.000       1.000        1.000
speechiness        0.000       0.036       0.049       0.084        0.965
acousticness       0.000       0.017       0.169       0.598        0.996
instrumentalness   0.000       0.000       0.000       0.049        1.000
liveness           0.000       0.098       0.132       0.273        1.000
valence            0.000       0.260       0.464       0.683        0.995
tempo              0.000      99.219  

In [6]:
import re
import unicodedata

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

sp_names = {norm(a) for v in spotify["artists"].dropna() for a in v.split(";")}

def parts(s):
    pieces = re.split(r"\s*(?:&|,|\bfeaturing\b|\bfeat\.?\b|\band\b)\s*", s, flags=re.I)
    return [norm(p) for p in pieces if p.strip()]

gr = grammy["artist"].dropna().drop_duplicates()
full_match = gr.map(lambda s: norm(s) in sp_names)
any_match = gr.map(lambda s: norm(s) in sp_names or any(p in sp_names for p in parts(s)))

res = pd.DataFrame({
    "metrica": ["artistas Grammy distintos (artist no nulo)",
                "con coincidencia por nombre completo",
                "con coincidencia por nombre completo o por partes"],
    "valor": [len(gr), int(full_match.sum()), int(any_match.sum())],
})
res["pct"] = (res["valor"] / len(gr) * 100).round(2)
res.to_csv(EVIDENCE / "profiling_artist_match.csv", index=False)
print(res.to_string(index=False))
print()
print("Ejemplos sin coincidencia:", list(gr[~any_match].head(15)))

                                          metrica  valor    pct
       artistas Grammy distintos (artist no nulo)   1658 100.00
             con coincidencia por nombre completo    544  32.81
con coincidencia por nombre completo o por partes    852  51.39

Ejemplos sin coincidencia: ['Christian Scott aTunde Adjuah', 'Theo Croker', 'Mark Guiliana', 'Lettuce', 'Bones UK', 'Brittany Howard', 'Karen O & Danger Mouse', 'Rival Sons', 'Big Thief', 'India.Arie', 'PJ Morton Featuring Jazmine Sullivan', 'Steve Lacy', 'Georgia Anne Muldrow', 'Ella Mai', 'PJ Morton']


In [7]:
g = grammy.copy()
null_art = g["artist"].isna()
from_workers = g["workers"].str.extract(r"\(([^()]+)\)\s*$")[0]
is_new_artist = g["category"].str.contains("New Artist", case=False, na=False)

rec_workers = null_art & from_workers.notna()
rec_new = null_art & is_new_artist & g["nominee"].notna()
rec = rec_workers | rec_new

out = pd.DataFrame({
    "metrica": ["artist nulo",
                "recuperable desde workers (texto entre paréntesis)",
                "recuperable desde nominee (Best New Artist)",
                "recuperable total",
                "sin recuperar"],
    "filas": [int(null_art.sum()), int(rec_workers.sum()), int(rec_new.sum()),
              int(rec.sum()), int((null_art & ~rec).sum())],
})
out["pct_de_nulos"] = (out["filas"] / null_art.sum() * 100).round(2)
out.to_csv(EVIDENCE / "profiling_artist_recoverable.csv", index=False)
print(out.to_string(index=False))


                                           metrica  filas  pct_de_nulos
                                       artist nulo   1840        100.00
recuperable desde workers (texto entre paréntesis)   1333         72.45
       recuperable desde nominee (Best New Artist)     65          3.53
                                 recuperable total   1398         75.98
                                     sin recuperar    442         24.02
